In [2]:
!python --version

Python 3.13.13


In [3]:
!uv pip install "opencv-python>=4.11,<5" ultralytics matplotlib tf-keras deepface


Checked 5 packages in 1.97s


In [4]:
import cv2
import ultralytics
from ultralytics import YOLO
import matplotlib.pyplot as plt
from deepface import DeepFace

In [9]:
DB_PATH = "my_db"
camera_stream_url = "http://192.168.1.26:8080/video"
name = "prottoy"

### Step 1: Set Up the Database Structure

Organize a folder on your computer named `my_db`. For each person you want to register, create a subfolder with their name containing their photo(s):

```text
my_db/
├── John/
│   ├── john_1.jpg
│   └── john_2.jpg
├── Jane/
│   ├── jane_1.jpg
│   └── jane_2.jpg

```


In [6]:
import os
import cv2

def enroll_face(person_name, image_frame, db_path="my_db"):
    
    user_folder = os.path.join(db_path, person_name)
    os.makedirs(user_folder, exist_ok=True)

    
    image_count = len(os.listdir(user_folder)) + 1
    file_path = os.path.join(user_folder, f"{person_name}_{image_count}.jpg")
    cv2.imwrite(file_path, image_frame)

    # IMPORTANT: Delete existing representation pickle files so DeepFace 
    # rebuilds its database index to include the new person.
    for file in os.listdir(db_path):
        if file.endswith(".pkl"):
            os.remove(os.path.join(db_path, file))

    return f"Enrolled {person_name} successfully! Saved to {file_path}"


# Enroll Face

In [ ]:
model = YOLO('yolov8n.pt')

cap = cv2.VideoCapture(camera_stream_url)

while True:
    ret, frame = cap.read()
    if not ret:
        print("Failed to grab frame")
        raise Exception("Failed to grab frame")

    text = enroll_face(name, frame)
    cv2.putText(frame, text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    frame = model(frame, verbose=False)[0].plot()
    cv2.imshow("Camera Stream", frame)
    
    if cv2.waitKey(100) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

In [10]:
DeepFace.find(
    img_path=os.path.join(DB_PATH, name, f"{name}_1.jpg"),
    db_path=DB_PATH, 
    model_name="Facenet512", 
    detector_backend="opencv",
    enforce_detection=False
)

26-08-13 06:22:02 - Found 66 newly added image(s), 0 removed image(s), 0 replaced image(s).


Finding representations: 100%|██████████| 66/66 [00:54<00:00,  1.22it/s]


26-08-13 06:22:57 - There are now 66 representations in ds_model_facenet512_detector_opencv_aligned_normalization_base_expand_0.pkl
26-08-13 06:22:57 - Searching my_db\prottoy\prottoy_1.jpg in 66 length datastore
26-08-13 06:22:58 - find function duration 56.40412759780884 seconds


[                        identity                                      hash  \
 0    my_db\prottoy\prottoy_1.jpg  31bbd6f59d2385c7e90b11ef90ffb62267247032   
 1    my_db\prottoy\prottoy_4.jpg  0a56ea9cb5e2d5b23c83559b0029e9df1b3bfd40   
 2    my_db\prottoy\prottoy_2.jpg  da4feff082246bdea6e8acd538679a86bfeb535b   
 3    my_db\prottoy\prottoy_5.jpg  80d4e8d4d8df41de56ef597c4174408117f17b22   
 4    my_db\prottoy\prottoy_6.jpg  bc623b978a02e9f74315c1ee622e14b044abf178   
 ..                           ...                                       ...   
 60  my_db\prottoy\prottoy_21.jpg  a372a93abe51ba4f80b1e545c29182625eb0f115   
 61   my_db\prottoy\prottoy_8.jpg  5f00f7e8354e7a2f05d3f26cd248a71253f142f1   
 62  my_db\prottoy\prottoy_16.jpg  8fb2b1739d3c9b66849c6f43a742d94aa39467a0   
 63  my_db\prottoy\prottoy_43.jpg  de72fae9a5b6fca2bb4f5e5f51b606caa95965c0   
 64  my_db\prottoy\prottoy_44.jpg  df236b9c50a5b629a1ad9b5ba2431b5a18094901   
 
     target_x  target_y  target_w  target_h  thres

In [ ]:
results = DeepFace.find(
    img_path="test_images\\ibrahim.png",
    db_path=DB_PATH,
    model_name="Facenet512",
    detector_backend="opencv",
    enforce_detection=False,
)

distance = results[0]['distance'][0]
matched_path = results[0]['identity'][0]
person_name = os.path.basename(os.path.dirname(matched_path))
print(person_name, distance, matched_path)

26-08-01 01:35:18 - Searching test_images\ibrahim.png in 126 length datastore
26-08-01 01:35:18 - find function duration 0.29755568504333496 seconds
ibrahim 0.188875 my_db\ibrahim\Screenshot 2026-08-01 012730.png


# Recognize Face

In [ ]:
import cv2
from deepface import DeepFace

cap = cv2.VideoCapture(camera_stream_url)

frame_count = 0
last_detected_name = "Searching..."

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_count += 1

    if frame_count % 15 == 0:
        try:
            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = DeepFace.find(
                img_path=rgb_frame,
                db_path=DB_PATH,
                model_name="Facenet512",
                detector_backend="opencv",
                enforce_detection=False,
            )

            if len(results) > 0 and not results[0].empty:
                matched_path = results[0]['identity'][0]
                person_name = os.path.basename(os.path.dirname(matched_path))
                distance = results[0]['distance'][0]
                last_detected_name = person_name if distance < 0.6 else "Unknown"

            else:
                last_detected_name = "Unknown"

        except Exception as e:
            last_detected_name = "No Face Detected"
            import traceback; traceback.print_exc()

    cv2.putText(frame, f"Person: {last_detected_name}", (30, 50),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    

    cv2.imshow("DeepFace Live Recognition", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


26-08-01 01:10:50 - Searching [[[188 188 184]
  [188 188 184]
  [188 188 184]
  ...
  [110 105  85]
  [108 104  84]
  [108 104  84]]

 [[188 188 184]
  [188 188 184]
  [188 188 184]
  ...
  [110 105  85]
  [109 105  85]
  [109 105  85]]

 [[188 188 184]
  [188 188 184]
  [188 188 184]
  ...
  [112 107  87]
  [110 106  86]
  [110 106  86]]

 ...

 [[ 18  18  18]
  [ 18  18  18]
  [ 18  18  18]
  ...
  [ 57  86  77]
  [ 50  88  76]
  [ 50  88  76]]

 [[ 18  18  18]
  [ 18  18  18]
  [ 18  18  18]
  ...
  [ 48  91  77]
  [ 43  92  76]
  [ 43  92  76]]

 [[ 18  18  18]
  [ 18  18  18]
  [ 18  18  18]
  ...
  [ 48  91  77]
  [ 43  92  76]
  [ 43  92  76]]] in 121 length datastore
26-08-01 01:10:50 - find function duration 0.9006974697113037 seconds
26-08-01 01:10:51 - Searching [[[206 206 202]
  [206 206 202]
  [206 206 202]
  ...
  [177 172 156]
  [196 191 175]
  [213 208 192]]

 [[206 206 202]
  [206 206 202]
  [206 206 202]
  ...
  [178 173 157]
  [194 189 173]
  [208 203 187]]

 [[206 2

# Test different backends

In [ ]:
from time import time


backends = ['opencv', 'ssd', 'dlib', 'mtcnn', 'retinaface', 'mediapipe']

for backend in backends:    
    try:
        start_time = time()

        face = DeepFace.extract_faces(
            img_path="my_db/prottoy/prottoy_1.jpg",
            detector_backend=backend
        )

        end_time = time()
        print(f"Backend: {backend}, Detected Faces: {len(face)}")
        print(f"Backend: {backend}, Analysis: {end_time - start_time:.4f} seconds")

    except Exception as e:
        print(f"Backend: {backend}, Error: {e}")
        continue


Backend: opencv, Detected Faces: 2
Backend: opencv, Analysis: 0.6827 seconds
Backend: ssd, Error: Face could not be detected in my_db/prottoy/prottoy_1.jpg.Please confirm that the picture is a face photo or consider to set enforce_detection param to False.
Backend: dlib, Error: Dlib is an optional detector, ensure the library is installed. Please install using 'pip install dlib'
Backend: mtcnn, Detected Faces: 1
Backend: mtcnn, Analysis: 2.0152 seconds
Backend: retinaface, Detected Faces: 1
Backend: retinaface, Analysis: 10.0973 seconds
Backend: mediapipe, Error: MediaPipe is an optional detector, ensure the library is installed. Please install using 'pip install mediapipe'
